In [9]:
import lxml.html
import requests
import time
import uuid
import pandas as pd

In [10]:
def make_request(url):
    user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    acc = 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8'
    headers = {
    'User-Agent': user_agent,
    'Accept': acc,
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    }

    resp = requests.get(url, headers = headers)
    print(f"Scraped with status code {resp.status_code}")
    print(f"Response headers: {resp.headers}")

    if resp.status_code == 429:
        time.sleep(2.5)
        resp = make_request(url)
    elif resp.status_code == 200:
        return resp

In [11]:
def scrape_judge(url):
    j_page = lxml.html.fromstring(make_request(url).text)
    r_d = {}

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//h3/text()"
    item_titles = j_page.xpath(xp1 + xp2)

    xp1 = "//div[@class = 'judge-info']//div[@class = 'about-list']"
    xp2 = "//div[@class = 'about-list-item']//p/text()"
    items = j_page.xpath(xp1 + xp2)

    for i, title in enumerate(item_titles):
        r_d[title.lower()] = items[i].replace("\t", "").strip().lower()

    return r_d

In [12]:
url = "https://state-law-research.org/state-justices/"

root = lxml.html.fromstring(make_request(url).text)

xp1 = "//section[@filter = 'judge']//"
xp2 = "a[contains(@href, 'judge')]//@href"
judge_links = root.xpath(xp1 + xp2)

xp1 = "//section[@filter = 'judge']//a[contains(@href, 'judge')]//"
xp2 = "div[@class = 'module--content module--content-post']"
xp3 = "//h2[@class = 'title']/text()"
judge_names = root.xpath(xp1 + xp2 + xp3)

judge_pd = {
    "JID": [],
    "name": []
}

Scraped with status code 200
Response headers: {'Date': 'Mon, 13 Apr 2026 23:34:41 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding, Accept-Encoding, Accept-Encoding, Accept-Encoding,Cookie', 'Link': '<https://state-law-research.org/wp-json/>; rel="https://api.w.org/", <https://state-law-research.org/wp-json/wp/v2/pages/2560>; rel="alternate"; title="JSON"; type="application/json", <https://state-law-research.org/?p=2560>; rel=shortlink', 'X-Powered-By': 'WP Engine', 'X-Cacheable': 'SHORT', 'Cache-Control': 'max-age=600, must-revalidate', 'X-Cache': 'HIT: 1', 'X-Cache-Group': 'normal', 'cf-cache-status': 'DYNAMIC', 'Set-Cookie': '__cf_bm=WHVDlolYn1yN6NPH1VXEr8phF1Hy6lvotDdyMfdJsRY-1776123281-1.0.1.1-HNZ.LclY29ixIUWCl_Y6QLYyowkNdxoADQMLSLt9UBL59rium5As9w7xRftncXo2hKCEBpUpJbtWNGdLQbY4mBNPXX_eXGNqxMOumMlNC0s; path=/; expires=Tue, 14-Apr-26 00:04:41 GMT; domain=.state-law-resea

In [13]:
for i, name in enumerate(judge_names):
    judge_pd["name"].append(name)
    judge_pd["JID"].append(uuid.uuid4())

    judge_dic = scrape_judge(judge_links[i])

    if judge_dic == {}:
        break

    for item in judge_dic.keys():
        data = judge_dic[item]
        if data is None:
            data
        if (item not in judge_pd):
            judge_pd[item] = [judge_dic[item]]
        else:
            judge_pd[item].append(judge_dic[item])

Scraped with status code 200
Response headers: {'Date': 'Mon, 13 Apr 2026 23:34:50 GMT', 'Content-Type': 'text/html; charset=UTF-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Content-Encoding': 'gzip', 'Vary': 'Accept-Encoding, Accept-Encoding, Accept-Encoding, Accept-Encoding,Cookie', 'Link': '<https://state-law-research.org/wp-json/>; rel="https://api.w.org/", <https://state-law-research.org/wp-json/wp/v2/judge/1944>; rel="alternate"; title="JSON"; type="application/json", <https://state-law-research.org/?p=1944>; rel=shortlink', 'X-Powered-By': 'WP Engine', 'X-Cacheable': 'SHORT', 'Cache-Control': 'max-age=600, must-revalidate', 'X-Cache': 'HIT: 1', 'X-Cache-Group': 'normal', 'cf-cache-status': 'DYNAMIC', 'Set-Cookie': '__cf_bm=r1UzYVfdiOGGYKOsOVX1M2S_rfgWJOPgCu3rLR53fMk-1776123290-1.0.1.1-BG3hFUGezL_QyZCU38ISSIfHJsVG0BmGqKFm.qELP3iRUV4hZJxyHx4TTBvL7TOYReomMEYYAmOuMJ8tgxgGfzqBxpUY3poDCLl9cW8G6qU; path=/; expires=Tue, 14-Apr-26 00:04:50 GMT; domain=.state-law-resea

In [14]:
for key in judge_pd.keys():
    print(f"{key}: {len(judge_pd[key])}")

JID: 345
name: 345
gender: 345
party: 343
race: 345
professional experience: 345
election type: 345
term start: 345
term end: 335
next election date: 323


In [15]:
del judge_pd["party"]
del judge_pd["next election date"]
del judge_pd["term end"]

In [16]:
pd.DataFrame(judge_pd)

,JID,name,gender,race,professional experience,election type,term start
0,c8c0b5c6-b65f-4b23-b4fe-620051a29a85,Abigail LeGrow,female,white,"corporate, judicial","appointed, leg confirmed","may 3, 2023"
1,e315e57b-41af-4457-b8b7-505d7280ba28,Adam Tanenbaum,male,white,"government civil (non civil rights), judicial,...",appointed,"january 14, 2026"
2,368be00a-8d69-4ea2-bce0-532bea2e1106,Aimee A. Oravec,female,white,"corporate, misc. private practice",appointed,"january 31, 2025"
3,96ce476e-503a-40d6-a82b-e4c9fe8d548e,Alisa Kelli Wise,female,white,"corporate, misc. private practice, judicial, o...","elected, partisan",2011
4,0230909e-0aa4-49ee-b97d-8e79ca44a4f8,Allison Riggs,female,white,"legal aid/non-govt civil rights, judicial",appointed,"september 11, 2023"
...,...,...,...,...,...,...,...
340,a5f01328-5e4d-413c-9c7e-cc6a7a301f54,"William J. Musseman, Jr.",male,white,"prosecutor, judicial",appointed,2022
341,3420010b-1adf-471c-adbf-7749b68877bb,William P. Robinson III,male,white,misc. private practice,"appointed, leg confirmed",2004
342,ae608ce8-a161-4e58-af7a-1de6306fb602,William R. Wooton,male,white,"prosecutor, plaintiff-side civil private pract...","elected, nonpartisan","january 1, 2021"
343,8a214921-87d4-4aec-ab72-3892062e087c,"William W. Hood, III",male,white,"prosecutor, corporate, plaintiff-side civil pr...","appointed, retention elected","january 13, 2014"
